# Lab 1.2: Multi-Turn Agent Loop Context Growth

**Day 1 - Topic 1**

Goal: measure how the output of six agent tool calls accumulates in the context window when the calls are unbounded versus scoped to source files.

> Open this notebook in Google Colab:
> [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/context-claude-companion/blob/main/day1/lab1_2_agent_loop_context_growth/start/lab1_2_agent_loop_context_growth.ipynb)

In [ ]:
# Run this cell once to install required packages.
# If running locally with the course venv already activated, you can skip this.
!pip install "pytest>=7" --quiet
# This lab also uses git and grep, which Colab provides.

In [ ]:
import os
import re
import subprocess
import sys
import tempfile
from pathlib import Path

def check(label: str, ok: bool) -> None:
    """Print PASS when a result is correct, TODO while a step is unfinished."""
    print(("PASS  " if ok else "TODO  ") + label)

## Section 1: Build the project and the runner

The helper builds a small telemetry-ingest project in a temporary folder (`src/`, `tests/`, a compiled `dist/` bundle, and a vendored `node_modules/` package), commits it with git, and makes one real edit so `git diff` has content. `run()` executes a shell command and returns everything it prints, as an agent's tool call would.

In [ ]:
def build_project(root: Path) -> None:
    files = {
        "CLAUDE.md": (
            "# Telemetry Worker\n"
            "- Edit Python sources in src/ only; never edit compiled files in dist/.\n"
            "- Run focused tests with: pytest tests/test_ingest.py -q\n"
        ),
        "pytest.ini": "[pytest]\npythonpath = .\n",
        "src/ingest/worker.py": "".join(
            f"def process_event_{i}(payload):\n    return payload.get('id')\n\n\n" for i in range(25)
        ),
        "src/ingest/validator.py": "".join(f"REQUIRED_FIELD_{i} = 'tenant_id'\n" for i in range(20)),
        "dist/ingest/worker.bundled.js": "".join(
            f"// compiled artifact line {i}: var process_event = function() {{}};\n" for i in range(400)
        ),
        "node_modules/vendor_pkg/index.js": "".join(
            f"// third-party vendor code line {i}{': process_event_wrapper' if i % 3 == 0 else ''}\n"
            for i in range(600)
        ),
        "tests/test_ingest.py": "from src.ingest import worker\n\n\n"
        + "".join(
            f"def test_case_{i}():\n    assert worker.process_event_{i % 25}({{'id': {i}}}) == {i}\n\n\n"
            for i in range(40)
        ),
    }
    for rel, text in files.items():
        path = root / rel
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(text)
    git = ["git", "-c", "user.name=demo", "-c", "user.email=demo@example.com"]
    subprocess.run(["git", "init", "-q"], cwd=root, check=True)
    subprocess.run(["git", "add", "-A"], cwd=root, check=True)
    subprocess.run(git + ["commit", "-qm", "baseline"], cwd=root, check=True)
    # The agent's actual edit, so `git diff` has real content.
    worker = root / "src/ingest/worker.py"
    worker.write_text(worker.read_text().replace("return payload.get('id')", "return payload.get('id') or ''", 1))


def run(root: Path, cmd: str) -> str:
    env = dict(os.environ, PYTHONDONTWRITEBYTECODE="1", PYTHONPATH="")
    proc = subprocess.run(cmd, shell=True, cwd=root, capture_output=True, text=True, env=env)
    # Normalise pytest's wall-clock time so repeated runs print identical numbers.
    return re.sub(r" in \d+\.\d+s", " in Ns", proc.stdout + proc.stderr)


def polluted_lines(output: str) -> int:
    """Count output lines that came from build output (dist/) or vendored code (node_modules/)."""
    markers = ("./dist/", "./node_modules/", "compiled artifact line", "third-party vendor code")
    return sum(1 for line in output.splitlines() if any(m in line for m in markers))


PYTEST = f"{sys.executable} -m pytest --color=no -p no:cacheprovider"

## Section 2: Write the scoped command plan

`UNBOUNDED` is what an agent does when nothing scopes it. Write `BOUNDED`: six commands that do the same job without dragging `dist/` or `node_modules/` into the window.

In [ ]:
UNBOUNDED = [
    ("cat CLAUDE.md", "cat CLAUDE.md"),
    ("ls -R (everything)", "ls -R"),
    ("grep -rn process_event . (whole repo)", "grep -rn process_event ."),
    ("cat dist/ingest/worker.bundled.js", "cat dist/ingest/worker.bundled.js"),
    ("pytest -vv (verbose dump)", f"{PYTEST} -vv"),
    ("re-read worker.py and dist bundle", "cat src/ingest/worker.py dist/ingest/worker.bundled.js"),
]

BOUNDED = [
    ("cat CLAUDE.md", "cat CLAUDE.md"),
    ("find src tests -type f", "find src tests -type f"),
    ("grep -rn -m5 process_event src/ingest", "grep -rn -m5 process_event src/ingest"),
    ("cat src/ingest/worker.py", "cat src/ingest/worker.py"),
    ("pytest tests/test_ingest.py -q", f"{PYTEST} tests/test_ingest.py -q"),
    ("git diff --stat src/", "git diff --stat src/"),
]

## Section 3: Measure each turn

Each command's output is added to the window and stays there, so the cost is cumulative. Tokens are estimated as characters divided by 4.

In [ ]:
def measure(root: Path, steps: list[tuple[str, str]]) -> list[dict]:
    """Run each command and track cumulative lines and estimated tokens (characters // 4)."""
    total_chars = total_lines = 0
    rows = []
    for label, cmd in steps:
        out = run(root, cmd)
        lines, chars = len(out.splitlines()), len(out)
        total_lines += lines
        total_chars += chars
        rows.append({"label": label, "lines": lines, "cum_lines": total_lines,
                     "cum_tokens": total_chars // 4, "polluted": polluted_lines(out)})
    return rows

## Section 4: Compare the two plans

Run both plans against a fresh copy of the project and compare the totals after turn 6.

In [ ]:
workdir = tempfile.TemporaryDirectory()
root = Path(workdir.name).resolve() / "telemetry_worker"
root.mkdir()
build_project(root)
unbounded = measure(root, UNBOUNDED)
bounded = measure(root, BOUNDED)
workdir.cleanup()

if len(unbounded) == 6 and len(bounded) == 6:
    print(f"{'Turn':<4} | {'Mode':<9} | {'Command':<40} | {'+Lines':>6} | {'Cum lines':>9} | {'Cum est tokens':>14} | Build/vendor lines")
    print("-" * 112)
    for i, (u, b) in enumerate(zip(unbounded, bounded), start=1):
        for mode, r in (("unbounded", u), ("bounded", b)):
            print(f"{i:<4} | {mode:<9} | {r['label']:<40} | {r['lines']:>6,d} | {r['cum_lines']:>9,d} | "
                  f"{r['cum_tokens']:>14,d} | {r['polluted']:>3d}")
        print()

    fu, fb = unbounded[-1], bounded[-1]
    pct = (1.0 - fb["cum_tokens"] / fu["cum_tokens"]) * 100.0
    print(f"After turn 6: unbounded {fu['cum_lines']:,d} lines / ~{fu['cum_tokens']:,d} tokens; "
          f"bounded {fb['cum_lines']:,d} lines / ~{fb['cum_tokens']:,d} tokens ({pct:.1f}% smaller).")
else:
    print("measure() has not returned six rows per plan yet. Finish the TODOs above.")

# Expected (pytest -vv adds a few header lines that vary by version, so your numbers differ slightly):
# After turn 6: unbounded ~1,660 lines / ~31,000 tokens; bounded ~127 lines / ~710 tokens (about 97.7% smaller).
print()
check("measure() returned one row per turn for both plans", len(unbounded) == 6 and len(bounded) == 6)
check("The bounded plan pulled in zero build or vendor lines",
      bool(bounded) and all(r["polluted"] == 0 for r in bounded))
check("The unbounded plan pulled in hundreds of build and vendor lines",
      bool(unbounded) and sum(r["polluted"] for r in unbounded) > 500)
check("The bounded plan ends under 5% of the unbounded token count",
      bool(bounded) and bool(unbounded) and bounded[-1]["cum_tokens"] < 0.05 * unbounded[-1]["cum_tokens"])

In [ ]:
print("Lab complete.")
print("Takeaway: Each turn adds its tool output to the window and keeps it there. Scoping greps to src/, reading only the file you will edit, and using pytest -q kept build and vendor files out of every later turn.")